# Filtro de Matrículas por Listado de Escuelas
Este notebook toma un archivo de matrículas a nivel nacional y lo filtra utilizando un listado específico de escuelas (en este caso, Arequipa). El cruce se realiza utilizando el `COD_MOD` (Código Modular), asegurándose de mantener los ceros a la izquierda y controlando que no haya códigos duplicados en el archivo de listado base. Adicionalmente, agrupa y suma los registros de matrícula de escuelas con múltiples filas antes de realizar el cruce.

In [ ]:
import pandas as pd
import os

# Rutas ajustadas asumiendo que el notebook está en notebooks/data_filtering/
listado_path = "../../data/cleaned/listado_escuelas_primaria_y_secundaria_arequipa.csv"
matricula_path = "../../data/cleaned/matricula_suma.csv"
output_path = "../../data/cleaned/matricula_final_arequipa_2025.csv"

In [ ]:
print("Cargando archivos CSV y forzando COD_MOD a string...")

# Carga de datos, forzando COD_MOD como string para no perder los ceros a la izquierda
listado = pd.read_csv(listado_path, dtype={'COD_MOD': str}, low_memory=False)
matricula = pd.read_csv(matricula_path, dtype={'COD_MOD': str}, low_memory=False)

print(f"Listado original: {len(listado)} filas")
print(f"Matricula original: {len(matricula)} filas")

In [ ]:
# Control de duplicados en el listado
duplicates = listado[listado.duplicated('COD_MOD', keep=False)]
if len(duplicates) > 0:
    print(f"ATENCIÓN: Se encontraron {len(duplicates)} filas con COD_MOD duplicado en el listado de escuelas.")
    print("Eliminando duplicados para mantener la consistencia...")
    listado = listado.drop_duplicates(subset=['COD_MOD'], keep='first')
else:
    print("Verificación exitosa: No hay COD_MOD duplicados en el listado de escuelas.")

In [ ]:
# Agrupar matrículas por COD_MOD sumando el total de matriculados y conservando el CODGEO
print(f"Filas en matricula_suma antes de agrupar: {len(matricula)}")
matricula_agrupada = matricula.groupby('COD_MOD', as_index=False).agg({
    'CODGEO': 'first',
    'total_matriculados_2025': 'sum'
})
print(f"Filas en matricula_suma después de agrupar: {len(matricula_agrupada)}")

# Filtrado por coincidencia 
valid_cod_mods = set(listado['COD_MOD'])
matricula_final = matricula_agrupada[matricula_agrupada['COD_MOD'].isin(valid_cod_mods)].copy()

print(f"Matricula filtrada final: {len(matricula_final)} filas únicas encontradas correspondientes al listado base.")

In [ ]:
# Exportar resultados
matricula_final.to_csv(output_path, index=False)
print(f"Guardado exitosamente en: {output_path}")

# Vista previa de los resultados
matricula_final.head()